# Reviewer 1 — Paired Block-Based Significance Tests

**Why not naive tests:** FCN3, GraphCast and HRES are scored on the same times,
stations and lead times, so their errors are paired, not independent samples (rules
out Mann-Whitney U). And station-hours aren't independent either — a single storm
inflates errors at all 12 stations at once, and consecutive hours at one station are
autocorrelated — so treating every station-hour as one independent observation
understates uncertainty (pseudo-replication), and treating the 12 stations as the only
independent unit (n=12) is conservative but throws away all temporal structure. Every
test below instead resamples or aggregates at the **block level**: a block is a run of
calendar days, all 12 stations and all lead times pooled together, so cross-station and
serial dependence within a block is preserved rather than assumed away.

**Three questions, one method family, one cell each:**
1. **Overall RMSE** — are FCN3/GraphCast/HRES significantly different, pooling
   2016-2022 and all lead times?
2. **High-wind RMSE & Bias** — same question, restricted to obs >= 10.8 m/s.
3. **Year-by-year skill vs HRES** — is each model's relative skill stable over
   2016-2022, or drifting?



In [7]:
import warnings
warnings.filterwarnings('ignore')

from pathlib import Path
from itertools import combinations

import numpy as np
import pandas as pd
from scipy.stats import wilcoxon
from IPython.display import display

OUT_ROOT      = Path('/cluster/work/projects/nn8106k/siyan/WF-experiments/case-study')
CSV_MASTER    = OUT_ROOT / 'master_forecast_dataset_72h_interp.csv'
CSV_HW_MERGED = OUT_ROOT / 'merged_fcn3_graphcast_hres_obs_all_years_72h_interp.csv'

YEARS         = [2016, 2017, 2018, 2019, 2020, 2021, 2022]
HIGH_WIND_THR = 10.8  # m/s -- 9-12 m/s bin (near-gale/gale, Beaufort 9-12)
MODEL_ORDER   = ['FCN3', 'GraphCast', 'HRES']
STATION_ORDER = ['SN88690', 'SN90490', 'SN90760', 'SN90800', 'SN90720', 'SN91740',
                  'SN89350', 'SN87110', 'SN85380', 'SN94500', 'SN96400', 'SN95350']

N_BOOT    = 10_000
BLOCK_LEN = 7  # days per resampled block (primary bootstrap)

print(f'YEARS={YEARS}  HIGH_WIND_THR={HIGH_WIND_THR} m/s  N_BOOT={N_BOOT:,}  BLOCK_LEN={BLOCK_LEN}d')

YEARS=[2016, 2017, 2018, 2019, 2020, 2021, 2022]  HIGH_WIND_THR=10.8 m/s  N_BOOT=10,000  BLOCK_LEN=7d


## Load Data

Both files are the same caches `rebuttal_model_comparison_analysis_72h_interpolate.ipynb`
builds (Sections 4 and 13) — read directly rather than rebuilt.


In [8]:
master_df = pd.read_csv(CSV_MASTER, parse_dates=['valid_time'])
master_long = master_df[['valid_time', 'station', 'model', 'forecast', 'obs', 'year']]
print(f'master_df: {len(master_df):,} rows, models={sorted(master_df.model.unique())}, '
      f'years={sorted(master_df.year.unique())}')

merged_all = pd.read_csv(CSV_HW_MERGED, parse_dates=['valid_time'])
MODEL_COLS = [('fcn3_wind', 'FCN3'), ('graphcast_wind', 'GraphCast'), ('hres_wind', 'HRES')]
hw_long = pd.concat([
    merged_all.loc[merged_all['obs_wind'] >= HIGH_WIND_THR,
                    ['station', 'valid_time', wind_col, 'obs_wind']]
        .rename(columns={wind_col: 'forecast', 'obs_wind': 'obs'})
        .assign(model=model)
    for wind_col, model in MODEL_COLS
], ignore_index=True)
print(f'High-wind subset (obs >= {HIGH_WIND_THR} m/s): {len(hw_long):,} rows')


master_df: 1,104,624 rows, models=['FCN3', 'GraphCast', 'HRES'], years=[np.int64(2016), np.int64(2017), np.int64(2018), np.int64(2019), np.int64(2020), np.int64(2021), np.int64(2022)]
High-wind subset (obs >= 10.8 m/s): 120,741 rows


## Shared Machinery


In [9]:
def make_cell_stats(df_long):
    """(date, station, model) -> n, sum_err, sum_sq_err."""
    df = df_long.dropna(subset=['forecast', 'obs']).copy()
    df['date'] = pd.to_datetime(df['valid_time']).dt.date
    df['err'] = df['forecast'] - df['obs']
    df['sq_err'] = df['err'] ** 2
    return (df.groupby(['date', 'station', 'model'])
              .agg(n=('err', 'size'), sum_err=('err', 'sum'), sum_sq_err=('sq_err', 'sum'))
              .reset_index())


def dense_arrays(stats, model, dates, station_order=STATION_ORDER):
    date_idx = {d: i for i, d in enumerate(dates)}
    station_idx = {s: j for j, s in enumerate(station_order)}
    sub = stats[stats['model'] == model]
    shape = (len(dates), len(station_order))
    out = {k: np.zeros(shape) for k in ('n', 'sum_err', 'sum_sq_err')}
    di, sj = sub['date'].map(date_idx).to_numpy(), sub['station'].map(station_idx).to_numpy()
    for k in out:
        out[k][di, sj] = sub[k].to_numpy()
    return out


def metric_from_sums(sums, metric):
    n = sums['n']
    with np.errstate(invalid='ignore', divide='ignore'):
        if metric == 'RMSE':
            return np.sqrt(sums['sum_sq_err'] / n)
        if metric == 'Bias':
            return sums['sum_err'] / n
    raise ValueError(metric)


def moving_block_indices(n_dates, block_len, n_boot, rng):
    """n_boot x (n_blocks*block_len) day-index array: draws n_blocks =
    ceil(n_dates/block_len) start points with replacement, each expanded into
    `block_len` consecutive days, circularly wrapped at the end of the date range."""
    n_blocks = int(np.ceil(n_dates / block_len))
    starts = rng.integers(0, n_dates, size=(n_boot, n_blocks))
    offsets = np.arange(block_len)
    idx = (starts[:, :, None] + offsets[None, None, :]) % n_dates
    return idx.reshape(n_boot, n_blocks * block_len)


def block_bootstrap_diff(arrays_a, arrays_b, metric, n_dates, block_len=BLOCK_LEN,
                          n_boot=N_BOOT, seed=0):
    """Primary test: moving block bootstrap 95% CI on mean_a(metric) - mean_b(metric),
    resampling `block_len`-consecutive-day blocks with replacement."""
    rng = np.random.default_rng(seed)
    idx = moving_block_indices(n_dates, block_len, n_boot, rng)
    boot_a = {k: v[idx].sum(axis=1) for k, v in arrays_a.items()}
    boot_b = {k: v[idx].sum(axis=1) for k, v in arrays_b.items()}
    diff = (np.nanmean(metric_from_sums(boot_a, metric), axis=1)
            - np.nanmean(metric_from_sums(boot_b, metric), axis=1))

    obs_a = {k: v.sum(axis=0, keepdims=True) for k, v in arrays_a.items()}
    obs_b = {k: v.sum(axis=0, keepdims=True) for k, v in arrays_b.items()}
    obs_diff = float(np.nanmean(metric_from_sums(obs_a, metric))
                      - np.nanmean(metric_from_sums(obs_b, metric)))

    ci_low, ci_high = np.nanpercentile(diff, [2.5, 97.5])
    p = float(min(1.0, 2 * min(np.nanmean(diff <= 0), np.nanmean(diff >= 0))))
    return obs_diff, ci_low, ci_high, p


def week_blocks(dates):
    """Non-overlapping 7-day blocks by real calendar offset from the first date (not
    sorted index), so a missing day doesn't shift later blocks. Returns a list of
    index arrays into `dates`."""
    dates_dt = pd.to_datetime(dates)
    week_id = (dates_dt - dates_dt.min()).days // 7
    blocks = {}
    for i, w in enumerate(week_id):
        blocks.setdefault(int(w), []).append(i)
    return list(blocks.values())


def weekly_wilcoxon(arrays_a, arrays_b, metric, blocks):
    """Secondary test: paired Wilcoxon signed-rank across non-overlapping weekly
    block differences (H0: median block difference = 0). No resampling."""
    diffs = []
    for idxs in blocks:
        idxs = np.array(idxs)
        sums_a = {k: v[idxs].sum(axis=0, keepdims=True) for k, v in arrays_a.items()}
        sums_b = {k: v[idxs].sum(axis=0, keepdims=True) for k, v in arrays_b.items()}
        diffs.append(np.nanmean(metric_from_sums(sums_a, metric))
                      - np.nanmean(metric_from_sums(sums_b, metric)))
    d = np.array(diffs)
    d = d[~np.isnan(d)]
    stat, p = wilcoxon(d)
    return d, float(p)


## Question 1 — Overall RMSE, All Years & Lead Times Pooled (vs Station Obs)

**Reviewer 1:** *"The paper uses a station-level cluster bootstrap with 10,000
resamples and only 12 stations. The number of bootstrap iterations is large, however,
the effective number of spatial clusters remains only 12, whereas the forecast errors
also have strong temporal correlation and shared weather-event dependence across
stations."*



In [10]:
stats_overall = make_cell_stats(master_long)
dates_overall = np.array(sorted(stats_overall['date'].unique()))
n_dates_overall = len(dates_overall)
arrays_overall = {m: dense_arrays(stats_overall, m, dates_overall) for m in MODEL_ORDER}
weeks_overall = week_blocks(dates_overall)

print(f'{n_dates_overall} calendar days -> {len(weeks_overall)} non-overlapping weekly blocks\n')

q1_rows = []
for i, (m1, m2) in enumerate(combinations(MODEL_ORDER, 2)):
    obs, ci_low, ci_high, p_boot = block_bootstrap_diff(
        arrays_overall[m1], arrays_overall[m2], 'RMSE', n_dates_overall, seed=1000 + i)
    d, p_wil = weekly_wilcoxon(arrays_overall[m1], arrays_overall[m2], 'RMSE', weeks_overall)
    sig_boot, sig_wil = bool(ci_low > 0 or ci_high < 0), bool(p_wil < 0.05)
    q1_rows.append({'model_a': m1, 'model_b': m2, 'mean_rmse_diff': obs,
                     'block_boot_ci_low': ci_low, 'block_boot_ci_high': ci_high,
                     'block_boot_p': p_boot, 'block_boot_significant': sig_boot,
                     'n_weekly_blocks': len(d), 'weekly_wilcoxon_p': p_wil,
                     'weekly_wilcoxon_significant': sig_wil,
                     'methods_agree': sig_boot == sig_wil})
    print(f'{m1} - {m2}: {obs:+.4f} m/s   {BLOCK_LEN}-day block-bootstrap 95% CI '
          f'[{ci_low:+.4f}, {ci_high:+.4f}]  p={p_boot:.4g} ({"sig" if sig_boot else "n.s."})'
          f'   |   weekly Wilcoxon p={p_wil:.4g} ({"sig" if sig_wil else "n.s."})')

q1_summary = pd.DataFrame(q1_rows)
display(q1_summary.round(4))


2557 calendar days -> 366 non-overlapping weekly blocks

FCN3 - GraphCast: +0.0287 m/s   7-day block-bootstrap 95% CI [+0.0164, +0.0408]  p=0 (sig)   |   weekly Wilcoxon p=2.472e-10 (sig)
FCN3 - HRES: +0.0730 m/s   7-day block-bootstrap 95% CI [+0.0568, +0.0886]  p=0 (sig)   |   weekly Wilcoxon p=9.084e-25 (sig)
GraphCast - HRES: +0.0443 m/s   7-day block-bootstrap 95% CI [+0.0304, +0.0576]  p=0 (sig)   |   weekly Wilcoxon p=2.491e-15 (sig)


,model_a,model_b,mean_rmse_diff,block_boot_ci_low,block_boot_ci_high,block_boot_p,block_boot_significant,n_weekly_blocks,weekly_wilcoxon_p,weekly_wilcoxon_significant,methods_agree
0,FCN3,GraphCast,0.0287,0.0164,0.0408,0.0,True,366,0.0,True,True
1,FCN3,HRES,0.0730,0.0568,0.0886,0.0,True,366,0.0,True,True
2,GraphCast,HRES,0.0443,0.0304,0.0576,0.0,True,366,0.0,True,True


## Question 2 — High-Wind RMSE & Bias (obs >= 10.8 m/s)


In [11]:
stats_hw = make_cell_stats(hw_long)
dates_hw = np.array(sorted(stats_hw['date'].unique()))
n_dates_hw = len(dates_hw)
arrays_hw = {m: dense_arrays(stats_hw, m, dates_hw) for m in MODEL_ORDER}
weeks_hw = week_blocks(dates_hw)

print(f'{n_dates_hw} high-wind calendar days -> {len(weeks_hw)} non-overlapping weekly blocks\n')

q2_rows = []
for mi, metric in enumerate(['RMSE', 'Bias']):
    print(f'--- {metric} ---')
    for i, (m1, m2) in enumerate(combinations(MODEL_ORDER, 2)):
        obs, ci_low, ci_high, p_boot = block_bootstrap_diff(
            arrays_hw[m1], arrays_hw[m2], metric, n_dates_hw, seed=2000 + 10 * mi + i)
        d, p_wil = weekly_wilcoxon(arrays_hw[m1], arrays_hw[m2], metric, weeks_hw)
        sig_boot, sig_wil = bool(ci_low > 0 or ci_high < 0), bool(p_wil < 0.05)
        q2_rows.append({'metric': metric, 'model_a': m1, 'model_b': m2, 'mean_diff': obs,
                         'block_boot_ci_low': ci_low, 'block_boot_ci_high': ci_high,
                         'block_boot_p': p_boot, 'block_boot_significant': sig_boot,
                         'n_weekly_blocks': len(d), 'weekly_wilcoxon_p': p_wil,
                         'weekly_wilcoxon_significant': sig_wil,
                         'methods_agree': sig_boot == sig_wil})
        print(f'  {m1} - {m2}: {obs:+.4f}   {BLOCK_LEN}-day block-bootstrap 95% CI '
              f'[{ci_low:+.4f}, {ci_high:+.4f}]  p={p_boot:.4g} ({"sig" if sig_boot else "n.s."})'
              f'   |   weekly Wilcoxon p={p_wil:.4g} ({"sig" if sig_wil else "n.s."})')

q2_summary = pd.DataFrame(q2_rows)
display(q2_summary.round(4))


1912 high-wind calendar days -> 358 non-overlapping weekly blocks

--- RMSE ---
  FCN3 - GraphCast: -0.2432   7-day block-bootstrap 95% CI [-0.2864, -0.1974]  p=0 (sig)   |   weekly Wilcoxon p=1.518e-46 (sig)
  FCN3 - HRES: -0.2816   7-day block-bootstrap 95% CI [-0.3359, -0.2165]  p=0 (sig)   |   weekly Wilcoxon p=4.713e-22 (sig)
  GraphCast - HRES: -0.0384   7-day block-bootstrap 95% CI [-0.0931, +0.0210]  p=0.2074 (n.s.)   |   weekly Wilcoxon p=1.11e-07 (sig)
--- Bias ---
  FCN3 - GraphCast: +0.4495   7-day block-bootstrap 95% CI [+0.4030, +0.4935]  p=0 (sig)   |   weekly Wilcoxon p=7.008e-57 (sig)
  FCN3 - HRES: +0.5172   7-day block-bootstrap 95% CI [+0.4497, +0.5749]  p=0 (sig)   |   weekly Wilcoxon p=1.848e-38 (sig)
  GraphCast - HRES: +0.0677   7-day block-bootstrap 95% CI [+0.0058, +0.1263]  p=0.0322 (sig)   |   weekly Wilcoxon p=2.546e-08 (sig)


,metric,model_a,model_b,mean_diff,block_boot_ci_low,block_boot_ci_high,block_boot_p,block_boot_significant,n_weekly_blocks,weekly_wilcoxon_p,weekly_wilcoxon_significant,methods_agree
0,RMSE,FCN3,GraphCast,-0.2432,-0.2864,-0.1974,0.0000,True,358,0.0,True,True
1,RMSE,FCN3,HRES,-0.2816,-0.3359,-0.2165,0.0000,True,358,0.0,True,True
2,RMSE,GraphCast,HRES,-0.0384,-0.0931,0.0210,0.2074,False,358,0.0,True,False
3,Bias,FCN3,GraphCast,0.4495,0.4030,0.4935,0.0000,True,358,0.0,True,True
4,Bias,FCN3,HRES,0.5172,0.4497,0.5749,0.0000,True,358,0.0,True,True
5,Bias,GraphCast,HRES,0.0677,0.0058,0.1263,0.0322,True,358,0.0,True,True


## Question 3 — Year-by-Year Relative Skill vs HRES (Temporal Generalization)

**Reviewer:** *"Could the authors report confidence intervals for the difference
between training-period and post-training-period performance, or yearly relative
skill against HRES? Such an analysis would better separate temporal distribution
shift from changes in annual forecast difficulty."*



In [17]:
q3_rows = []
for mi, model in enumerate(['FCN3', 'GraphCast']):
    for i, yr in enumerate(YEARS):
        df_year = master_long[master_long['year'] == yr]
        stats_year = make_cell_stats(df_year)
        dates_year = np.array(sorted(stats_year['date'].unique()))
        arrays_m = dense_arrays(stats_year, model, dates_year)
        arrays_h = dense_arrays(stats_year, 'HRES', dates_year)

        obs_diff, ci_low, ci_high, p = block_bootstrap_diff(
            arrays_m, arrays_h, 'RMSE', len(dates_year), seed=3000 + 100 * mi + i)
        sig = bool(ci_low > 0 or ci_high < 0)
        q3_rows.append({'model': model, 'year': yr, 'rmse_diff_vs_hres': obs_diff,
                         'ci_low': ci_low, 'ci_high': ci_high, 'p_value': p,
                         'significant_0.05': sig})

q3_summary = pd.DataFrame(q3_rows)
print(f'Yearly RMSE difference vs HRES (RMSE_model - RMSE_HRES), {BLOCK_LEN}-day block '
      'bootstrap 95% CI (positive = model worse than HRES that year):')
display(q3_summary.round(4))


Yearly RMSE difference vs HRES (RMSE_model - RMSE_HRES), 7-day block bootstrap 95% CI (positive = model worse than HRES that year):


,model,year,rmse_diff_vs_hres,ci_low,ci_high,p_value,significant_0.05
0,FCN3,2016,0.0225,-0.0155,0.0597,0.2434,False
1,FCN3,2017,0.0332,-0.0212,0.0830,0.2148,False
2,FCN3,2018,0.1121,0.0822,0.1421,0.0000,True
3,FCN3,2019,0.1102,0.0731,0.1468,0.0000,True
4,FCN3,2020,0.0758,0.0398,0.1115,0.0000,True
5,FCN3,2021,0.0661,0.0264,0.1047,0.0014,True
6,FCN3,2022,0.0921,0.0486,0.1357,0.0002,True
7,GraphCast,2016,0.0183,-0.0135,0.0474,0.2438,False
8,GraphCast,2017,0.0284,-0.0162,0.0694,0.2026,False
9,GraphCast,2018,0.0735,0.0495,0.0975,0.0000,True
